In [ ]:
from sklearn.model_selection import KFold, cross_validate, RepeatedKFold
from sklearn.preprocessing import PowerTransformer,  StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LinearRegression, HuberRegressor

from ml_enhance import CorrelationFilter

from sklearn import pipeline
import pandas as pd
import numpy as np
import json

In [139]:
def make_pipeline(model, step_name):
    return pipeline.Pipeline([
        ("variance", VarianceThreshold(threshold=0.0)),
        ("remove_corr", CorrelationFilter(threshold=0.95)),
        ("transform", PowerTransformer(method='yeo-johnson', standardize=False)),
        ("scale", StandardScaler()),
        (step_name, model)
    ])

In [ ]:
pl_linear = make_pipeline(LinearRegression(), "predict")
pl_huber = make_pipeline(HuberRegressor(), "predict")

As the data (even after transformation and scaling) consists of a lot of outliers that cannot be removed (as they are valid molecules and are therefore part of the real world observables) I decided to use [Huber regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.HuberRegressor.html) instead of linear regression, this method is more robust to outliers.

In [141]:
df = pd.read_csv("../data/processed_dataset_wo_metals_w_even_more_qm.csv")

In [ ]:
y = df["solubility"]
X = df.drop(["solubility", "smiles", "canon_smiles", "id"], axis=1)

In [ ]:
X = X.drop([
        "avg_atomic_quadrupole_principal_invariant_3", # quadrupole principal invariant 3 features correlate highly with the invariant 2 features, so can drop them
        "max_atomic_quadrupole_principal_invariant_3",
        "molecular_quadrupole_principal_invariant_3",
        "avg_atomic_dipole_dipole_interaction" # the dipole dipole interaction between atoms would physically not be that influential on the solubility, can drop it
    ], axis=1)

In [144]:
with open("../data/rdkit_feature_names.json", "r") as f:
    rdkit_feature_list: list = json.load(f)

mask = X.columns.isin(rdkit_feature_list)

In [145]:
X_topo = X.iloc[:, mask]
X_qm = X.iloc[:, ~mask]

## Huber Regression: Topology VS. QM + Topology

In [167]:
inner_kf = KFold(n_splits=5, shuffle=True, random_state=42)
repeated_kf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=15)

In [168]:
scoring = {
    "r2": "r2",
    "MSE": "neg_mean_squared_error"
}

In [169]:
param_grid = {
    "predict__epsilon": [1.1, 1.35, 1.5, 2.0],
    "predict__alpha": [1e-5, 1e-4, 1e-3, 1e-2]
}

In [171]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=pl_huber,
    param_grid=param_grid,
    cv=inner_kf,
    scoring='r2',
    n_jobs=4,
    verbose=11
)

In [ ]:
scores_combo = cross_validate(grid, X, y, cv=repeated_kf, scoring=scoring, n_jobs=4, return_estimator=True, return_train_score=True)

In [ ]:
scores_topo = cross_validate(grid, X_topo, y, cv=repeated_kf, scoring=scoring, n_jobs=4, return_estimator=True, return_train_score=True)

In [ ]:
scores_qm = cross_validate(grid, X_qm, y, cv=repeated_kf, scoring=scoring, n_jobs=4, return_estimator=True, return_train_score=True)

In [156]:
print(f"Train R2 scores:\nTopology alone: {scores_topo["train_r2"].mean()}\nQM alone: {scores_qm["train_r2"].mean()}\nCombined: {scores_combo["train_r2"].mean()}")
print("\n")
print(f"Test R2 scores:\nTopology alone: {scores_topo["test_r2"].mean()}\nQM alone: {scores_qm["test_r2"].mean()}\nCombined: {scores_combo["test_r2"].mean()}")

Train R2 scores:
Topology alone: 0.8262991760437502
QM alone: 0.775008823707611
Combined: 0.8440770885813674


Test R2 scores:
Topology alone: 0.8149067153443121
QM alone: 0.7620393936183701
Combined: 0.8239383800925194


In [164]:
from scipy.stats import ttest_rel

ttest_result = ttest_rel(scores_combo["test_r2"], scores_topo["test_r2"])

print("Topo mean R2:", scores_topo["test_r2"].mean())
print("Combined mean R2:", scores_combo["test_r2"].mean())
print("Mean improvement:", (scores_combo["test_r2"] - scores_topo["test_r2"]).mean())
print(f"p-value: {ttest_result[1]} ->{'not' if ttest_result[1] > 0.05 else ''} statistically significant")

Topo mean R2: 0.8149067153443121
Combined mean R2: 0.8239383800925194
Mean improvement: 0.009031664748207158
p-value: 0.0031675029535858596 -> statistically significant


In [159]:
print(f"Train MSE scores:\nTopology alone: {np.abs(scores_topo["train_MSE"]).mean()}\nQM alone: {np.abs(scores_qm["train_MSE"]).mean()}\nCombined: {np.abs(scores_combo["train_MSE"]).mean()}")
print("\n")
print(f"Test MSE scores:\nTopology alone: {np.abs(scores_topo["test_MSE"]).mean()}\nQM alone: {np.abs(scores_qm["test_MSE"]).mean()}\nCombined: {np.abs(scores_combo["test_MSE"]).mean()}")

Train MSE scores:
Topology alone: 0.9251251696815433
QM alone: 1.1982908401673646
Combined: 0.8304457321503875


Test MSE scores:
Topology alone: 0.9848945239449116
QM alone: 1.2657923782562783
Combined: 0.9369128569120904


In [163]:
ttest_result = ttest_rel(np.abs(scores_combo["test_MSE"]), np.abs(scores_topo["test_MSE"]))

print("Topo mean R2:", np.abs(scores_topo["test_MSE"]).mean())
print("Combined mean R2:", np.abs(scores_combo["test_MSE"]).mean())
print("Mean improvement:", (np.abs(scores_combo["test_MSE"]) - np.abs(scores_topo["test_MSE"])).mean())
print(f"p-value: {ttest_result[1]} ->{'not' if ttest_result[1] > 0.05 else ''} statistically significant")

Topo mean R2: 0.9848945239449116
Combined mean R2: 0.9369128569120904
Mean improvement: -0.04798166703282142
p-value: 0.0035364071333895833 -> statistically significant


Based on the results i got from the Huber regression, the QM descriptors alone seem to give the worst performance out of the three and **there seems to be a small significant difference between the topological descriptors alone and the combined feature set**. The combined set of features provides a slightly better prediction. Whether this is truely because of the QM descriptors, or because of model bias should be analyzed by looking at other models (RF, GAM, KRR?)